# MP1 — Additional Charts

This notebook produces the additional charts for the MP1 analysis of PurpleAir sensor data.
Each chart answers a different analytical question from the MP1a declaration and is saved as a `.png` file.

In [1]:
# Setup
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "requests", "python-dotenv"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        install(pkg)

import os
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://api.purpleair.com/v1"
HEADERS = {"X-API-Key": API_KEY}

print("Setup complete.")

Setup complete.



[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Fetch outdoor Seattle sensors (same bounding box as starter notebook)
fields = ["name", "latitude", "longitude", "pm2.5", "pm2.5_10minute", "pm2.5_60minute", "pm2.5_24hour", "temperature", "humidity"]

outdoor_params = {
    "fields": ",".join(fields),
    "max_age": 3600,
    "location_type": 0,
    "nwlng": -122.45, "nwlat": 47.70,
    "selng": -122.25, "selat": 47.55,
}

r = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=outdoor_params)
r.raise_for_status()
raw = r.json()
df_outdoor = pd.DataFrame(raw["data"], columns=raw["fields"])
df_outdoor["sensor_type"] = "Outdoor"
print(f"Outdoor sensors: {len(df_outdoor)}")

Outdoor sensors: 138


In [3]:
# Fetch indoor Seattle sensors
indoor_params = {
    "fields": ",".join(fields),
    "max_age": 3600,
    "location_type": 1,
    "nwlng": -122.45, "nwlat": 47.70,
    "selng": -122.25, "selat": 47.55,
}

r = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=indoor_params)
r.raise_for_status()
raw_indoor = r.json()
df_indoor = pd.DataFrame(raw_indoor["data"], columns=raw_indoor["fields"])
df_indoor["sensor_type"] = "Indoor"
print(f"Indoor sensors: {len(df_indoor)}")

# Combine and remove physically implausible outliers (>200 µg/m³)
df_combined = pd.concat(
    [df_outdoor[["sensor_type", "pm2.5", "pm2.5_24hour"]],
     df_indoor[["sensor_type", "pm2.5", "pm2.5_24hour"]]],
    ignore_index=True
)
df_combined = df_combined[df_combined["pm2.5"] <= 200].copy()
print(f"Total clean sensors: {len(df_combined)}")

Indoor sensors: 123
Total clean sensors: 257


---

## Chart 1 — Indoor vs Outdoor PM2.5

**Question:** How does air quality within Seattle differ between indoor and outdoor sensors?

Medians are used instead of means because a small number of malfunctioning sensors inflate mean values significantly. The 24-hour average is included alongside the current reading to smooth out any momentary spikes.

In [4]:
# Summarize by sensor type using median (robust to outliers)
summary = (
    df_combined.groupby("sensor_type")
    .agg(
        median_pm25=("pm2.5", "median"),
        median_pm25_24hr=("pm2.5_24hour", "median"),
    )
    .reset_index()
    .round(2)
)

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    name="Current reading",
    x=summary["sensor_type"],
    y=summary["median_pm25"],
    marker_color=["#4C9BE8", "#55A868"],
    text=summary["median_pm25"],
    texttemplate="%{text:.1f}",
    textposition="outside",
))

fig2.add_trace(go.Bar(
    name="24-hour average",
    x=summary["sensor_type"],
    y=summary["median_pm25_24hr"],
    marker_color=["#9ECAE1", "#A1D99B"],
    text=summary["median_pm25_24hr"],
    texttemplate="%{text:.1f}",
    textposition="outside",
))

fig2.update_layout(
    title="Seattle indoor sensors consistently read lower PM2.5 than outdoor sensors",
    xaxis_title="Sensor Location",
    yaxis_title="Median PM2.5 (µg/m³)",
    barmode="group",
    height=450,
    yaxis=dict(range=[0, 7]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=20, r=20, t=80, b=50),
)

fig2.show()
fig2.write_image("chart_indoor_vs_outdoor.png")
print("Saved chart_indoor_vs_outdoor.png")

Saved chart_indoor_vs_outdoor.png


**Chart rationale:**

A grouped bar chart lets readers compare two values (current vs. 24-hour average) side by side for each sensor type simultaneously. Medians are used rather than means because a handful of faulty sensors with implausible readings (>200 µg/m³) were filtered, and medians are robust to any remaining skew. The result is clear: indoor sensors in Seattle report meaningfully lower PM2.5 than outdoor sensors — roughly half the level on a current reading — which suggests buildings act as an effective particulate barrier under normal (non-wildfire) conditions.

In [5]:
# Fetch outdoor sensors for 8 major U.S. cities
cities = {
    "Seattle":       dict(nwlng=-122.45, nwlat=47.70, selng=-122.25, selat=47.55),
    "Los Angeles":   dict(nwlng=-118.50, nwlat=34.20, selng=-118.10, selat=33.90),
    "San Francisco": dict(nwlng=-122.55, nwlat=37.85, selng=-122.35, selat=37.70),
    "New York City": dict(nwlng=-74.10,  nwlat=40.80, selng=-73.70,  selat=40.60),
    "Chicago":       dict(nwlng=-87.80,  nwlat=42.00, selng=-87.55,  selat=41.75),
    "Denver":        dict(nwlng=-105.10, nwlat=39.80, selng=-104.85, selat=39.60),
    "Phoenix":       dict(nwlng=-112.15, nwlat=33.65, selng=-111.85, selat=33.35),
    "Houston":       dict(nwlng=-95.60,  nwlat=29.90, selng=-95.20,  selat=29.60),
}

city_results = []
for city, bbox in cities.items():
    params = {
        "fields": "pm2.5,pm2.5_24hour",
        "max_age": 3600,
        "location_type": 0,
        **bbox,
    }
    r = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=params)
    if r.status_code != 200:
        print(f"  {city}: API error {r.status_code}")
        continue
    raw = r.json()
    city_df = pd.DataFrame(raw["data"], columns=raw["fields"])
    if city_df.empty:
        print(f"  {city}: no sensors found")
        continue
    city_results.append({
        "city": city,
        "sensors": len(city_df),
        "median_pm25": round(city_df["pm2.5"].median(), 2),
        "median_pm25_24hr": round(city_df["pm2.5_24hour"].median(), 2),
    })
    print(f"  {city}: {len(city_df)} sensors, median PM2.5 = {city_df['pm2.5'].median():.2f}")

city_df_summary = pd.DataFrame(city_results).set_index("city")
city_df_summary

  Seattle: 138 sensors, median PM2.5 = 2.70


  Los Angeles: 362 sensors, median PM2.5 = 4.45


  San Francisco: 231 sensors, median PM2.5 = 3.00


  New York City: 52 sensors, median PM2.5 = 3.00


  Chicago: 29 sensors, median PM2.5 = 2.10


  Denver: 23 sensors, median PM2.5 = 2.30


  Phoenix: 21 sensors, median PM2.5 = 0.70


  Houston: 31 sensors, median PM2.5 = 13.50


,sensors,median_pm25,median_pm25_24hr
city,,,
Seattle,138,2.70,4.90
Los Angeles,362,4.45,17.50
San Francisco,231,3.00,5.10
New York City,52,3.00,7.75
Chicago,29,2.10,4.40
Denver,23,2.30,3.60
Phoenix,21,0.70,2.90
Houston,31,13.50,14.80


---

## Chart 2 — City-by-City PM2.5 Comparison

**Question:** How does Seattle compare for air quality compared to other major U.S. cities?

Median PM2.5 is used rather than mean because several cities (Phoenix, Seattle, Los Angeles) have sensors with implausible spike readings that would inflate the mean. The EPA 'Good' threshold (12 µg/m³) is shown as a reference line.

In [6]:
city_plot = city_df_summary.reset_index().sort_values("median_pm25", ascending=True)

fig3 = px.bar(
    city_plot,
    x="median_pm25",
    y="city",
    orientation="h",
    color="median_pm25",
    color_continuous_scale="RdYlGn_r",
    text="median_pm25",
    title="Seattle's air quality ranks among the best of eight major U.S. cities",
    labels={
        "median_pm25": "Median PM2.5 (µg/m³)",
        "city": "",
    },
)

fig3.update_traces(texttemplate="%{text:.1f} µg/m³", textposition="outside")

# EPA 'Good' threshold reference line
fig3.add_vline(
    x=12,
    line_dash="dash",
    line_color="gray",
    annotation_text="EPA 'Good' limit (12 µg/m³)",
    annotation_position="top right",
)

fig3.update_layout(
    coloraxis_showscale=False,
    xaxis_title="Median PM2.5 (µg/m³)",
    height=480,
    margin=dict(l=20, r=120, t=60, b=40),
)

fig3.show()
fig3.write_image("chart_city_comparison.png")
print("Saved chart_city_comparison.png")

Saved chart_city_comparison.png


**Chart rationale:**

A horizontal bar chart works well for ranked categorical comparisons — city names are long enough to crowd a vertical x-axis, and sorting from cleanest to worst makes the ranking immediately readable. The red-yellow-green color scale maps intuitively to air quality without requiring the reader to interpret the numbers first. The dashed EPA 'Good' threshold (12 µg/m³) anchors the chart in a real-world standard: every city except Houston falls well within that limit, and Seattle sits near the cleaner end of the distribution alongside San Francisco and Chicago.